In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, RepeatedStratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_selection import mutual_info_classif
from xgboost import XGBClassifier

# 1. Load Data
target_name = 'Survived'
train_url = '/kaggle/input/competitions/titanic/train.csv'
test_url = '/kaggle/input/competitions/titanic/test.csv'

X_full = pd.read_csv(train_url)
X_test_full = pd.read_csv(test_url)

y = X_full[target_name].copy()
X_full.drop(columns=[target_name], inplace=True)

test_passenger_ids = X_test_full['PassengerId'].copy()

# 2. Base Data Cleaning
X_test_full['Fare'] = X_test_full['Fare'].fillna(X_full['Fare'].median())
X_full['Embarked'] = X_full['Embarked'].fillna(X_full['Embarked'].mode()[0])
X_test_full['Embarked'] = X_test_full['Embarked'].fillna(X_full['Embarked'].mode()[0])

# A. Titles Extraction
for df in [X_full, X_test_full]:
    df['Title'] = df['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
    df['Title'] = df['Title'].replace(['Mlle', 'Ms'], 'Miss')
    df['Title'] = df['Title'].replace('Mme', 'Mrs')    
    rare_titles = ['Lady', 'Countess', 'Capt', 'Col', 'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona']
    df['Title'] = df['Title'].replace(rare_titles, 'Rare')

# B. Age Imputation based on Train Titles
title_age_means = X_full.groupby('Title')['Age'].mean()

for df in [X_full, X_test_full]:
    for title, mean_age in title_age_means.items():
        df.loc[(df['Age'].isna()) & (df['Title'] == title), 'Age'] = mean_age
    
    # Is_Child feature (< 10 years old)
    df['Is_Child'] = (df['Age'] < 10).astype(int)
    
    # Helper feature for Group Survival Strategy
    df['Is_Woman_Or_Child'] = ((df['Sex'] == 'female') | (df['Is_Child'] == 1)).astype(int)

    # Family Features
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['Is_Alone'] = (df['FamilySize'] == 1).astype(int)

    # Log Fare Transformation
    df['Fare_Per_Person'] = df['Fare'] / df['FamilySize']
    df['LogFare_Per_Person'] = np.log1p(df['Fare_Per_Person'])

    # Extract Deck
    df['Deck'] = df['Cabin'].astype(str).str[0]
    df['Deck'] = df['Deck'].replace({'n': 'U', 'T': 'U'})

# C. Advanced Feature: Woman/Child Ticket Group Survival Strategy
# نربط بيانات الـ Train بالـ Test لاستخراج المجموعات بدون تسريب الـ Target
df_train_temp = X_full.copy()
df_train_temp['Survived'] = y
df_test_temp = X_test_full.copy()
df_test_temp['Survived'] = np.nan

df_all = pd.concat([df_train_temp, df_test_temp], ignore_index=True)

# حساب نجاة المجموعات بناءً على النساء والأطفال في الـ Train حصراً
df_all['Group_Survival'] = 0.5  # قيمة محايدة للأفراد أو المجموعات بدون بيانات نجاة

for ticket, group in df_all.groupby('Ticket'):
    if len(group) > 1:
        # البحث عن النساء والأطفال المتواجدين في الـ Train للمجموعة
        w_c = group[(group['Is_Woman_Or_Child'] == 1) & (group['Survived'].notna())]
        if len(w_c) > 0:
            survival_rate = w_c['Survived'].mean()
            df_all.loc[df_all['Ticket'] == ticket, 'Group_Survival'] = survival_rate

# إعادة تحديث المتغير في البيانات الأصلية
X_full['Group_Survival'] = df_all.iloc[:len(X_full)]['Group_Survival'].values
X_test_full['Group_Survival'] = df_all.iloc[len(X_full):]['Group_Survival'].values

print("Clean feature engineering with Group Survival Strategy done!")

# 3. Explicit Feature Selection
categorical_cols = ['Sex', 'Embarked', 'Title', 'Deck']
numerical_cols = ['Pclass', 'Is_Child', 'Is_Alone', 'FamilySize', 'LogFare_Per_Person', 'Group_Survival']

my_cols = categorical_cols + numerical_cols

X_train_clean = X_full[my_cols].copy()
X_test_clean = X_test_full[my_cols].copy()

# Train/Validation Split for Tuning
X_train_sub, X_valid_sub, y_train_sub, y_valid_sub = train_test_split(
    X_train_clean, y, train_size=0.8, test_size=0.2, random_state=0, stratify=y
)

# 4. Pipeline Setup
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, numerical_cols),
    ('cat', cat_transformer, categorical_cols)
])

base_model = XGBClassifier(
    random_state=0,
    eval_metric='logloss'
)

my_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', base_model)
])

# 5. Calculating & Graphing Mutual Information (MI)
print('--- Calculating & Plotting Mutual Information ---')

X_train_transformed = preprocessor.fit_transform(X_train_sub)
feature_names = preprocessor.get_feature_names_out()

mi_scores = mutual_info_classif(X_train_transformed, y_train_sub, random_state=0)
mi_series = pd.Series(mi_scores, index=feature_names).sort_values(ascending=True)

plt.figure(figsize=(10, max(4, round(mi_series.count() * 0.35))), dpi=100)
mi_series.plot(kind='barh', color='skyblue', edgecolor='black')
plt.title('Mutual Information Scores (Including Group Survival)', fontsize=12, pad=15)
plt.xlabel('MI Score', fontsize=10)
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# 6. Hyperparameter Tuning with GridSearchCV
param_grid = {
    'model__n_estimators': [50, 100, 150],
    'model__max_depth': [3, 4],
    'model__learning_rate': [0.01, 0.03, 0.05],
    'model__subsample': [0.7, 0.8, 1.0],
    'model__colsample_bytree': [0.7, 0.8, 1.0]
}

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=0)

grid_search = GridSearchCV(
    estimator=my_pipeline, 
    param_grid=param_grid, 
    cv=cv, 
    scoring='accuracy', 
    n_jobs=-1, 
    verbose=1
)

print('Searching for best parameters ...')
grid_search.fit(X_train_sub, y_train_sub)

print('--- Tuning Result ---')
print('Best Parameters:', grid_search.best_params_)
print('Best Validation Accuracy:', grid_search.best_score_)

# 7. Final Fit on Full Dataset with Best Parameters & Predict
best_pipeline = grid_search.best_estimator_
best_pipeline.fit(X_train_clean, y)

preds_test = best_pipeline.predict(X_test_clean)

# 8. Submission File
output = pd.DataFrame({
    'PassengerId': test_passenger_ids, 
    target_name: preds_test
})

output.to_csv('submission.csv', index=False)
print("Your submission with Group Survival Strategy was successfully saved!")